# 10 — Values and Contextual Embeddings

**Description:** Add value projections to attention weights, compute weighted sums, and build a complete single-head attention operation.
**Level:** Beginner
**Tags:** Language Models, Attention, Values, Contextual Embeddings, Weighted Sums

Notebook 09 produced an attention matrix $A$. Each row says how strongly one target position attends to every allowed source. But weights alone carry no content. This notebook introduces **values** and completes one attention head:

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(rac{QK^T}{\sqrt{d_k}}+Might)V$$

By the end, you will be able to explain values, compute contextual vectors as weighted sums, and inspect how every token representation changes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Rebuild the causal attention weights

The setup is unchanged from Notebooks 08–09. Keeping it fixed lets us focus entirely on what happens after the weights are known.

In [ ]:
tokens = ["the", "robot", "fixed", "it"]
X = np.array([[1.0, 0.0, 0.2], [0.2, 1.0, 0.6], [0.1, 0.7, 1.0], [0.8, 0.2, 0.4]])
W_Q = np.array([[1.0, 0.0], [0.0, 0.8], [0.5, 0.5]])
W_K = np.array([[0.6, 0.2], [0.1, 1.0], [0.8, 0.3]])

def softmax(values, axis=-1):
    shifted = values - np.max(values, axis=axis, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=axis, keepdims=True)

Q, K = X @ W_Q, X @ W_K
scores = Q @ K.T / np.sqrt(Q.shape[-1])
mask = np.triu(np.ones(scores.shape, dtype=bool), k=1)
A = softmax(np.where(mask, -np.inf, scores), axis=-1)
print("A shape:", A.shape)
print(A)

## 2. Values describe the information to send

A third learned projection creates values:

$$V=XW_V$$

Keys participate in matching; values carry the payload. Separating them lets a token advertise one set of features for routing while sending another set of features as content.

In [ ]:
W_V = np.array([
    [1.0, 0.0, 0.5],
    [0.0, 1.0, 0.5],
    [0.5, -0.5, 1.0],
])
V = X @ W_V

print("X shape:  ", X.shape)
print("W_V shape:", W_V.shape)
print("V shape:  ", V.shape)
for token, value in zip(tokens, V):
    print(f"{token:>5}: {value}")

## 3. One target computes a weighted sum

For target position $i$, the contextual output is:

$$z_i=\sum_j A_{ij}v_j$$

The attention row supplies scalar mixing weights; the value rows supply vectors.

In [ ]:
target = tokens.index("it")
contributions = A[target, :, None] * V
context_for_it = contributions.sum(axis=0)

print("weights used by 'it':", A[target])
for source, weight, contribution in zip(tokens, A[target], contributions):
    print(f"{source:>5} × {weight:.3f} -> {contribution}")
print("context for 'it':", context_for_it)

The contributions are vectors, not token choices. Attention blends information continuously. A high weight makes a source value contribute more, but all allowed sources can contribute.

## 4. Compute every contextual vector at once

Matrix multiplication performs all weighted sums:

$$Z=AV$$

With $A:(T,T)$ and $V:(T,d_v)$, the result has shape $(T,d_v)$—one updated vector per target token.

In [ ]:
Z = A @ V

print("A shape:", A.shape)
print("V shape:", V.shape)
print("Z shape:", Z.shape)
print(Z)
assert np.allclose(Z[target], context_for_it)

### Inspect the causal boundary

The first token can attend only to itself, so its output must equal its own value. Later tokens mix progressively longer prefixes.

In [ ]:
print("first value: ", V[0])
print("first output:", Z[0])
assert np.allclose(V[0], Z[0])

for i, token in enumerate(tokens):
    used = [tokens[j] for j in range(len(tokens)) if A[i, j] > 0]
    print(f"{token:>5} can use: {used}")

## 5. Visualize representation changes

Our values and outputs are 3D. We can compare the first two coordinates to see each value vector and the contextual mixture produced for that position.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for i, token in enumerate(tokens):
    ax.scatter(V[i, 0], V[i, 1], color="#4C78A8")
    ax.scatter(Z[i, 0], Z[i, 1], color="#F58518")
    ax.annotate("", xy=Z[i, :2], xytext=V[i, :2], arrowprops={"arrowstyle": "->", "color": "gray"})
    ax.annotate(token, Z[i, :2], xytext=(5, 5), textcoords="offset points")
ax.scatter([], [], color="#4C78A8", label="value before mixing")
ax.scatter([], [], color="#F58518", label="contextual output")
ax.set(xlabel="feature 1", ylabel="feature 2", title="Attention mixes value vectors into contextual outputs")
ax.legend()
plt.show()

A contextual output lies in the weighted-average region of the values it can access because its weights are non-negative and sum to one. A later output can move differently even when its own value is similar, because its attention row is different.

## 6. Complete single-head attention

We can now package the full operation. The function returns both outputs and weights because inspecting attention patterns is useful for learning and debugging.

In [ ]:
def single_head_attention(X, W_Q, W_K, W_V, causal=True):
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    scores = Q @ K.T / np.sqrt(Q.shape[-1])
    if causal:
        future = np.triu(np.ones(scores.shape, dtype=bool), k=1)
        scores = np.where(future, -np.inf, scores)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

outputs, weights = single_head_attention(X, W_Q, W_K, W_V)
print("outputs:\n", outputs)
assert np.allclose(outputs, Z)
assert np.allclose(weights, A)

## 7. Separate routing from content

Changing $W_V$ changes the content delivered without changing attention weights. Changing $W_Q$ or $W_K$ changes routing. This division of labor is central to attention.

In [ ]:
W_V_alternative = np.array([[0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0]])
outputs_alt, weights_alt = single_head_attention(X, W_Q, W_K, W_V_alternative)

print("weights unchanged:", np.allclose(weights, weights_alt))
print("outputs unchanged:", np.allclose(outputs, outputs_alt))

## 8. Attention output versus residual update

$Z$ is the output of the attention head. Transformer blocks normally project it back to model width and add the original stream through a residual connection. We postpone that surrounding block structure until Notebook 11 so the weighted-value operation stays visible here.

## 9. Challenges

1. Manually reproduce the contextual output for `fixed`.
2. Replace one attention row with `[0, 0, 0, 1]`. Which value is copied?
3. Use `W_V = np.eye(3)`. What information is now being mixed?
4. Turn off causal masking and inspect how the first output changes.
5. Explain why $d_v$ does not have to equal $d_k$.

## Takeaways

- Values are $V=XW_V$: the information available to send.
- Queries and keys determine **where to read**; values determine **what is read**.
- Each contextual vector is a weighted sum of value vectors.
- The matrix product $AV$ updates every position at once.
- Scaling, masking, softmax, and weighted values together form a complete single attention head.
- Notebook 11 will run several heads in parallel and combine their outputs.